# Neural Network: Fashion-MNIST Classification

A project from my postgraduate course *Deep Learning and Neural Networks*, implementing a feedforward artificial neural network with configurable hidden layers from scratch, and using it to classify Fashion-MNIST images — first as a binary classifier, then extended to full 10-class classification, with hyperparameter tuning and a sigmoid vs. ReLU activation comparison along the way.

See the [main README](./README.md) for a summary of the approach and results.

## Artificial Neural Network Implementation

In [78]:
import numpy as np
import random
import matplotlib.pyplot as plt

class ANN:

    #==========================================#
    # The init method is called when an object #
    # is created. It can be used to initialize #
    # the attributes of the class.             #
    #==========================================#
    def __init__(self, no_inputs, hidden_layers=[], output_size = 1, max_iterations=20, learning_rate=0.1, activation = 'sigmoid',
                 batch_size = 100, classification_type = 'binary'):

        self.no_inputs = no_inputs
        self.hidden_layers = hidden_layers
        self.max_iter = max_iterations
        self.learning_rate = learning_rate
        self.classification_type = classification_type
        self.output_size = output_size
        self.activation = activation
        self.batch_size = batch_size
        
        # weights for each layer must be initialised as a matrix with:
        # one row for each node in layer k;
        # one column for each node in layer k - 1
        
        prev_nodes = self.no_inputs + 1  # initialising the number of inputs for the current layer
        self.hidden_weights = []         # initialise hidden weights
        
        for h in range(len(hidden_layers)):
            self.hidden_weights.append(
                np.random.randn(hidden_layers[h], prev_nodes) * np.sqrt(1 / prev_nodes)
            )
            prev_nodes = hidden_layers[h] + 1

        self.output_weights = np.random.randn(self.output_size, prev_nodes) * np.sqrt(1 / prev_nodes)

    # defining a function to check the dimensions of the matrices
    def print_weights(self):
        print("Hidden Layer Weights:")
        for idx, matrix in enumerate(self.hidden_weights):
            print(f"Layer {idx + 1} weights (shape: {matrix.shape}):")
            print(matrix)
            print()
        
        print("Output Layer Weights:")
        print(f"Output weights (shape: {self.output_weights.shape}):")
        print(self.output_weights)

    #===================================#
    # Performs the activation function. #
    # Expects an array of values of     #
    # shape (1,N) where N is the number #
    # of nodes in the layer.            #
    #===================================#
    
    # activation functions are implemented the same way as the perceptrons, although we will be dealing with an array of inputs
    # can use sigmoid, tanh, ReLU etc.
    
    def activate(self, a):
        if self.activation == 'sigmoid': # sigmoid activation function
            return 1 / (1 + np.exp(-a))
        if self.activation == 'ReLU': # ReLU activation function
            return np.maximum(0, a) 
            
    
    #=========================================#
    # Performs feed-forward prediction on     #
    # batch of inputs.                        #
    #=========================================#
    
    # prediction is fed forward through network, layer by layer.
    # each nodes output is some activation function applied on the outputs of the previous layers nodes with the current layers weights.
    
    def do_predict(self, x):
        last_layer_output = np.hstack((1, x))  # Add bias term
        activations = [last_layer_output]

        # Forward pass through hidden layers
        for k in range(len(self.hidden_layers)):
            output_layer_k = self.activate(np.matmul(self.hidden_weights[k], last_layer_output.T))
            last_layer_output = np.hstack((1, output_layer_k))  # Add bias for next layer
            activations.append(last_layer_output)

        # Final layer output
        final_layer_output = np.matmul(self.output_weights, last_layer_output.T)

        if self.classification_type == 'binary':
            final_layer_output = self.activate(final_layer_output) 
            predicted_class = 1 if final_layer_output >= 0.5 else 0  # Threshold for binary classification
        else:  # For multiclass classification
            predicted_class = np.argmax(final_layer_output)  # Select node with max value

        return final_layer_output, activations, predicted_class

    
    #============================================================#
    # Encoding labels                                            #
    # Simple bibary or one hot depending on classification type  #           
    #============================================================#
    
    # Re-labeling for the binary classification (same as perceptron)
    
    def label_to_binary(self, labels):
        label_mapping = {
            0: 0,  # T-shirt/top (cloth)
            1: 0,  # Trouser (cloth)
            2: 0,  # Pullover (cloth)
            3: 0,  # Dress (cloth)
            4: 0,  # Coat (cloth)
            5: 1,  # Sandal (shoe)
            6: 0,  # Shirt (cloth)
            7: 0,  # Sneaker (shoe)
            8: 0,  # Bag (cloth)
            9: 0   # Ankle_boot (shoe)
        }
        return [label_mapping[label] for label in labels]
    
    # Creating a one hot encoding function for the multiclass classification
    
    def label_to_one_hot(self, labels):
        num_classes = self.output_size
        one_hot_encoded = np.zeros((len(labels), num_classes))

        for i, label in enumerate(labels):
            label = int(label)
            if 0 <= label < num_classes:
                one_hot_encoded[i, label] = 1
            else:
                raise ValueError(f"Invalid label: {label}, expected range: 0 to {num_classes - 1}") # error if i forget to change output nodes

        return one_hot_encoded
    
    #=========================================#
    # Normalizing data                        #
    # so weights are evenly updated           #
    #=========================================#

    def min_max_normalize(self, inputs):
        inputs = np.array(inputs)
        min_vals = np.min(inputs, axis=0)
        max_vals = np.max(inputs, axis=0)
        return (inputs - min_vals) / (max_vals - min_vals)
        
    #===============================#
    # Trains the net using labelled #
    # training data.                #
    #===============================#
    def do_train(self, training_data):
        print('---------------')
        print("Training Neural Network....")
        
        for i in range(self.max_iter):
            np.random.shuffle(training_data) # shuffle data
            
            # split data into labels and inputs
            labels = training_data[:, 0]
            inputs = training_data[:, 1:]  
            
            # Normalize data
            inputs = self.min_max_normalize(inputs)
            
            # define number of batches from batch size
            num_batches = len(training_data) // self.batch_size
            
            # start of mini batch
            for j in range(num_batches):
                start = j * self.batch_size
                end = start + self.batch_size
                batch_inputs = inputs[start:end]
                batch_labels = labels[start:end]
                
                # Initialize gradients
                output_layer_gradients = []
                hidden_gradients = [[] for _ in range(len(self.hidden_weights))]
                
                for data, label in zip(batch_inputs, batch_labels):
                    # Forward pass
                    output, activations, _ = self.do_predict(data)
                    
                    # Backpropagation
                    
                    # creating an if function to calculate output error based on binary or multiclass
                    if self.classification_type == 'binary':
                        target_label = self.label_to_binary([label])[0] 
                        output_error = np.array([output - target_label]) 
                        partial_derivative = output_error * activations[-1]
                        
                    elif self.classification_type == 'multiclass':
                        target_label = self.label_to_one_hot([label])[0] 
                        output_error = np.array([output - target_label])
                        a = (activations[-1]).reshape(-1, 1)
                        partial_derivative = np.matmul(output_error.T, a.T)

                    # append the output layer gradients
                    output_layer_gradients.append(partial_derivative)
                    
                    # Backpropagate errors through hidden layers
                    layer_weights = self.hidden_weights + [self.output_weights]

                    next_layer_error = output_error
                    
                    # Employ gradient calculations depending on what activation function is used.
                    if self.activation == 'sigmoid':
                        for l in range(len(self.hidden_layers)):
                            if l == 0:
                                hidden_error = activations[-1][1:] * (1 - activations[-1][1:]) * (np.matmul(output_error, (self.output_weights)[:, 1:]))
                                hidden_gradients[0].append(activations[-2][:, np.newaxis] * hidden_error[np.newaxis, :].T)
                                next_layer_error = hidden_error
                            else:
                                hidden_error = activations[-(l+1)][1:] * (1 - activations[-(l+1)][1:]) * (np.matmul(next_layer_error, layer_weights[-(l+1)][:,1:]))
                                hidden_gradients[l].append(activations[-(l+2)][:, np.newaxis] * hidden_error[np.newaxis, :].T)
                                next_layer_error = hidden_error
                            
                    elif self.activation == 'ReLU':
                        for l in range(len(self.hidden_layers)):
                            if l == 0:
                                hidden_error = (activations[-1][1:] > 0) * (np.matmul(output_error, self.output_weights[:, 1:]))
                                hidden_gradients[0].append(activations[-2][:, np.newaxis] * hidden_error[np.newaxis, :].T)
                                next_layer_error = hidden_error
                            else:
                                hidden_error = (activations[-(l+1)][1:] > 0) * (np.matmul(next_layer_error, layer_weights[-(l+1)][:, 1:]))
                                hidden_gradients[l].append(activations[-(l+2)][:, np.newaxis] * hidden_error[np.newaxis, :].T)
                                next_layer_error = hidden_error
            
                self.output_weights = self.output_weights - self.learning_rate * np.mean(output_layer_gradients, axis = 0)
            
                # Compute the mean gradients for each hidden layer
                mean_hidden_gradients = [np.mean(layer_gradients, axis = 0) for layer_gradients in hidden_gradients]
                
                for i in range(len(self.hidden_layers)):
                    self.hidden_weights[i] = self.hidden_weights[i] - self.learning_rate * mean_hidden_gradients[len(self.hidden_layers) - 1 - i][:,:,0]
                    
        print("Neural Network trained!")
        print('---------------')
                

    #=========================================#
    # Tests the prediction on each element of #
    # the testing data. Prints the precision, #
    # recall, and accuracy.                   #
    #=========================================#
    def test(self, test_data):
        print('---------------')
        print('Testing Neural Network...')
        
        # Split the test data into labels and inputs
        labels = test_data[:, 0]
        inputs = test_data[:, 1:]
        
        # Normalize inputs
        inputs = self.min_max_normalize(inputs)

        # Initialize metrics
        correct_predictions = 0
        tp = fp = tn = fn = 0 

        predictions = []
        true_labels = []

        for data, label in zip(inputs, labels):
            # Forward pass
            output, activations, predicted_class = self.do_predict(data)

            # Convert label based on classification type
            if self.classification_type == 'binary':
                true_label = self.label_to_binary([label])[0]  # Convert to binary label
            else:
                true_label = label  # Keep integer label for multiclass

            predictions.append(predicted_class)
            true_labels.append(true_label)

            # Update confusion matrix and accuracy
            if predicted_class == true_label:
                correct_predictions += 1
                if self.classification_type == 'binary':
                    if predicted_class == 1:
                        tp += 1
                    else:
                        tn += 1
            else:
                if self.classification_type == 'binary':
                    if predicted_class == 1:
                        fp += 1
                    else:
                        fn += 1

        # Calculate accuracy
        accuracy = correct_predictions / len(labels)

        if self.classification_type == 'binary':
            # Binary classification metrics
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0

            # Print metrics
            print(f"Accuracy: {accuracy:.4f}")
            print(f"Precision: {precision:.4f}")
            print(f"Recall: {recall:.4f}")

            # Confusion Matrix
            confusion_matrix = np.array([[tp, fp], [fn, tn]])
            print("Confusion Matrix:")
            print(confusion_matrix)
        
        else:
            print(f"Multiclass Classification Accuracy: {accuracy:.4f}")

            # Confusion matrix for multiclass
            # tried to create confusion matrices with just numpy but wasnt formatting correctly
            # therefore have to use sklearn to plot confusion matrix
            
            from sklearn.metrics import confusion_matrix
            cm_multiclass = confusion_matrix(true_labels, predictions)
            print("Confusion Matrix:")
            print(cm_multiclass)

        print('Test Complete!')
        print('---------------')



## Loading the Data

In [2]:
data_path = "./"
train_data = np.loadtxt(data_path + 'mnist_fashion_train.csv', delimiter = ",")
test_data = np.loadtxt(data_path + "mnist_fashion_test.csv", delimiter = ",")

## Building the Network

In [72]:
net = ANN(28*28,                           # defining number of inputs
          hidden_layers=[3, 4, 5],         # Defining number of layers and number of nodes in each layer (3 layers with int value representing number of nodes)
          learning_rate = 0.1,             # Defining learning rate
          max_iterations = 20,             # defining number of iterations
          classification_type = 'binary',  # Defining the classification type
          output_size = 1,                 # defining number of nodes in output
          batch_size = 32                  # Defining the size of the mini batch
          ) 

## Training and Testing: Binary Classification

In [ ]:
net.do_train(train_data)

---------------
Training Neural Network....
Neural Network trained!
---------------


In [ ]:
net.test(test_data)

---------------
Testing Neural Network...
Accuracy: 0.9297
Precision: 0.6822
Recall: 0.5560
Confusion Matrix:
[[8741  259]
 [ 444  556]]
Test Complete!
---------------


We know the ANN is working, since it is producing an output, but as we can see, the results are quite poor. So I'm going to do a bit of hyperparameter tuning to fix this. I think the main values affecting the results will be the hidden layers, batch size, and learning rate. I'm not going to change the value of max iterations, as I believe lowering it would risk the ANN underfitting the data, while increasing it would also increase the training time, which is already fairly large.

In [75]:
net_2 = ANN(28*28,                         
          hidden_layers=[3, 4, 5],         
          learning_rate = 0.01,             
          max_iterations = 20,             
          classification_type = 'binary',  
          output_size = 1,                 
          batch_size = 64                 
          ) 

net_2.do_train(train_data)

net_2.test(test_data)

---------------
Training Neural Network....
Neural Network trained!
---------------
---------------
Testing Neural Network...
Accuracy: 0.9000
Precision: 0.0000
Recall: 0.0000
Confusion Matrix:
[[9000    0]
 [1000    0]]
Test Complete!
---------------


The learning rate is too low, and therefore hasn't had enough time to converge.

In [ ]:
net_3 = ANN(28*28,                         
          hidden_layers=[3, 4, 5],         
          learning_rate = 0.2,             
          max_iterations = 20,             
          classification_type = 'binary',  
          output_size = 1,                 
          batch_size = 64                 
          ) 

net_3.do_train(train_data)

net_3.test(test_data)

---------------
Training Neural Network....
Neural Network trained!
---------------
---------------
Testing Neural Network...
Accuracy: 0.9319
Precision: 0.7813
Recall: 0.4430
Confusion Matrix:
[[8876  124]
 [ 557  443]]
Test Complete!
---------------


Since I have to keep max_iterations at a lower value, this greatly affects what the learning rate can be — 0.01 was clearly too low, and going up to 0.2 may be too high, preventing the ANN from finding a minimum. A learning rate of 0.1 seems more optimal, and I'll increase the batch size a little more in hopes that this will also improve performance.

In [79]:
net_4 = ANN(28*28,                         
          hidden_layers=[3, 4, 5],         
          learning_rate = 0.1,             
          max_iterations = 20,             
          classification_type = 'binary',  
          output_size = 1,                 
          batch_size = 100                 
          ) 

net_4.do_train(train_data)

net_4.test(test_data)

---------------
Training Neural Network....
Neural Network trained!
---------------
---------------
Testing Neural Network...
Accuracy: 0.9864
Precision: 0.9364
Recall: 0.9270
Confusion Matrix:
[[ 927   63]
 [  73 8937]]
Test Complete!
---------------


This is much better! Since this is just a binary classification with a single output node, I don't think increasing the hidden layers further is very necessary right now, even though it would likely improve accuracy, since it would also increase training time. But when I move to training a multi-class classifier with 10 output nodes, I'd expect to need to increase these by roughly an order of magnitude.

In [83]:
net_5 = ANN(28*28,                         
          hidden_layers=[30, 40, 50],         
          learning_rate = 0.1,             
          max_iterations = 20,             
          classification_type = 'binary',  
          output_size = 1,                 
          batch_size = 100                 
          ) 

net_5.do_train(train_data)

net_5.test(test_data)

---------------
Training Neural Network....
Neural Network trained!
---------------
---------------
Testing Neural Network...
Accuracy: 0.9863
Precision: 0.9481
Recall: 0.9130
Confusion Matrix:
[[ 913   50]
 [  87 8950]]
Test Complete!
---------------


## Extending to Multi-Class Classification

I will now implement multi-class classification. As found above, my optimal learning rate and batch size are 0.1 and 100 respectively, with 20 max iterations.

In [29]:
net_multi = ANN(28*28,                   
          hidden_layers=[3,4,5],   
          learning_rate = 0.1,     
          max_iterations = 20,     
          classification_type= 'multiclass',
          output_size = 10, 
          batch_size = 100
          ) 

#net_multi.print_weights()

net_multi.do_train(train_data)

---------------
Training Neural Network....
Neural Network trained!
---------------


In [30]:
net_multi.test(test_data)

---------------
Testing Neural Network...
Multiclass Classification Accuracy: 0.3006
Confusion Matrix:
[[964  27   0   0   0   8   0   0   0   1]
 [ 33 967   0   0   0   0   0   0   0   0]
 [980  18   0   0   0   2   0   0   0   0]
 [789 209   0   0   0   2   0   0   0   0]
 [971  28   0   0   0   1   0   0   0   0]
 [  0   2   0   0   0  48   0 223   0 727]
 [967  29   0   0   0   4   0   0   0   0]
 [  0   0   0   0   0   0   0 358   0 642]
 [112 702   0   0   0 142   0   1   0  43]
 [  0   0   0   0   0   5   0 326   0 669]]
Test Complete!
---------------


We can see that the multi-class classification is working, but showing poor results. This is likely due to the hidden layer hyperparameter — with a more sophisticated classification task, the number of hidden layer nodes needs to reflect this. With 10 output nodes, the hidden layers should have at least on the order of 10s of nodes. A good rule of thumb is that the number of hidden nodes should be between the number of input and output nodes, giving a wide range from 10 to 784. That said, as seen earlier, it may not be feasible to use too many neurons, since it will take significantly longer to train.

In [31]:
net_multi_tuning = ANN(28*28,                   
          hidden_layers=[30,40,50],   
          learning_rate = 0.1,     
          max_iterations = 20,     
          classification_type= 'multiclass',
          output_size = 10, 
          batch_size = 100
          ) 

#net_multi_tuning.print_weights()

net_multi_tuning.do_train(train_data)

net_multi_tuning.test(test_data)

---------------
Training Neural Network....
Neural Network trained!
---------------
---------------
Testing Neural Network...
Multiclass Classification Accuracy: 0.7986
Confusion Matrix:
[[856   1  27  78   7   5   5   0  20   1]
 [  5 930  16  40   7   0   0   0   2   0]
 [ 22   1 760  17 171   3   9   0  16   1]
 [ 37   9  24 884  32   1   7   0   6   0]
 [  1   2 114  61 795   5  13   0   9   0]
 [  0   0   0   2   0 889   0  63   6  40]
 [270   1 240  66 295   5  83   1  39   0]
 [  0   0   0   0   0  34   0 902   0  64]
 [  4   1  17  10   2   3   5   6 951   1]
 [  0   0   0   0   0  14   0  49   1 936]]
Test Complete!
---------------


This is definitely a lot better than before, with an accuracy of 0.79, but I think it can be improved further. I'll run it one more time with slightly more nodes.

In [80]:
net_multi_tuning = ANN(28*28,                   
          hidden_layers=[80, 70, 60],   
          learning_rate = 0.1,     
          max_iterations = 20,     
          classification_type= 'multiclass',
          output_size = 10, 
          batch_size = 100
          ) 

# net_multi_tuning.print_weights()

net_multi_tuning.do_train(train_data)

net_multi_tuning.test(test_data)

---------------
Training Neural Network....
Neural Network trained!
---------------
---------------
Testing Neural Network...
Multiclass Classification Accuracy: 0.8142
Confusion Matrix:
[[804   2  27  70   4   4  66   0  22   1]
 [  2 946  14  26   9   0   1   0   2   0]
 [ 16   2 737   7 134   0  92   0  12   0]
 [ 28  20  23 845  28   1  50   0   5   0]
 [  1   3 116  40 706   1 124   0   9   0]
 [  0   0   0   2   0 873   0  71   8  46]
 [199   2 160  53 106   2 442   0  36   0]
 [  0   0   0   0   0  34   0 895   0  71]
 [  1   1  16   7   1   4  14   4 951   1]
 [  0   0   0   0   0  11   0  45   1 943]]
Test Complete!
---------------


## Activation Function Comparison

Comparing:
- Perceptron vs. neural network for binary classification, using sigmoid activation
- Sigmoid vs. ReLU activation for multi-class classification

In [ ]:
net_relu = ANN(28*28,         
          hidden_layers=[80, 70, 60],       
          learning_rate = 0.1,        
          max_iterations = 20,       
          classification_type = 'binary',
          activation = 'ReLU',
          output_size = 1,                 
          batch_size = 100
          ) 

net_relu.do_train(train_data)

net_relu.test(test_data)

---------------
Training Neural Network....
Neural Network trained!
---------------
---------------
Testing Neural Network...
Accuracy: 0.9923
Precision: 0.9685
Recall: 0.9540
Confusion Matrix:
[[ 954   31]
 [  46 8969]]
Test Complete!
---------------


In [ ]:
net_relu = ANN(28*28,         
          hidden_layers=[80, 70, 60],       
          learning_rate = 0.1,        
          max_iterations = 20,       
          classification_type = 'multiclass',
          activation = 'ReLU',
          output_size = 10,                 
          batch_size = 100
          ) 

net_relu.do_train(train_data)

net_relu.test(test_data)

---------------
Training Neural Network....
Neural Network trained!
---------------
---------------
Testing Neural Network...
Multiclass Classification Accuracy: 0.8776
Confusion Matrix:
[[807   3   8  26   4   1 138   0  13   0]
 [  4 967   2  21   3   0   2   0   1   0]
 [ 11   2 780  11  76   0 119   1   0   0]
 [ 17   9   9 884  34   1  41   1   4   0]
 [  0   1 110  34 775   0  73   0   7   0]
 [  0   0   0   0   0 950   0  32   2  16]
 [118   0  57  26  65   0 719   0  15   0]
 [  0   0   0   0   0  11   0 962   0  27]
 [  2   0   5   5   3   1   9   6 969   0]
 [  0   0   0   0   0   5   0  31   1 963]]
Test Complete!
---------------


## Binary Classification: Perceptron vs. Neural Network (Sigmoid)

When comparing the performance of the neural network and the perceptron for binary classification, both models performed well but with some key differences. The neural network achieved an accuracy of 0.9863, with a precision of 0.9481 and recall of 0.9130, with the confusion matrix indicating slightly more misclassifications than the perceptron. The perceptron, on the other hand, scored a higher accuracy (0.9929), precision (0.9813), and recall (0.947), with fewer misclassifications. However, we also need to consider training time for both models — the neural network took a much more significant amount of time to train compared to the perceptron. This makes the perceptron a more time-efficient model, especially if the goal is a quick binary classification with high accuracy. The neural network, while more time-consuming, may still offer better performance on more complex datasets.

## Multi-Class Classification: Sigmoid vs. ReLU

To keep the comparison fair, both networks are created with the same hyperparameters. The model using ReLU achieved a higher accuracy score (ReLU = 0.8776, Sigmoid = 0.8142), indicating better generalisation and learning capability. This is likely because ReLU minimizes the vanishing gradient problem that affects sigmoid, where extreme input values produce very small gradients and therefore slower learning — giving ReLU an advantage through more efficient backpropagation. The confusion matrices further highlight this, showing fewer misclassifications and a better distribution of correct predictions with ReLU. These results suggest ReLU is the more suitable activation function for classifying this Fashion-MNIST dataset.

## Design Choices

### Weight Initialisation

I implemented Xavier initialisation for this network, as it helps ensure stability across layers by keeping activations from being too large or too small, which would otherwise lead to exploding or vanishing gradients. This was achieved by initialising a normal distribution and scaling it down by multiplying by the square root of the inverse of the number of inputs for that layer.

### Number of Iterations

Due to computational power and training time, I set the number of iterations to 20. I believe this was the most optimal value — low enough to keep training time reasonable, but not so low that the networks wouldn't have enough time to learn meaningful patterns, or so high that it would risk overfitting or excessive training time for similar end results. 20 iterations provides a good balance between efficiency and performance, ensuring sufficient convergence while keeping computational cost low.

### Learning Rate

A larger learning rate allows the model to learn faster by making larger updates to the weights, reducing training time. However, if set too high, the network may 'overshoot' and fail to converge; similarly, if too small, the network may not converge fast enough and would need more iterations to compensate, increasing computational cost. From experimentation, with my chosen number of iterations, a learning rate of 0.1 was the most effective, allowing the model to make meaningful progress while avoiding excessively slow training.

### Layers

Since the dataset consists of greyscale 28x28 images with relatively simple patterns, I didn't believe a deep neural network was necessary for effective classification, so I settled on 3 hidden layers. The input layer has 784 nodes, and the output layer has 1 node for binary classification or 10 for multi-class. There's no fixed rule for the number of hidden layer nodes, but a good rule of thumb is for it to fall between the number of input and output nodes — a wide range given 784 inputs. I was heavily limited by computational power, though, so I settled on [80, 70, 60] nodes for the hidden layers, which I felt gave the best balance between good results and keeping computational cost low.